In [1]:

# Week 8 - Cloud Infrastructure Setup

import os
import sys
import platform
import psutil

print("Python version:", sys.version)
print("Operating system:", platform.platform())

print("\nCPU Information")
print("CPU cores:", os.cpu_count())
print("RAM:", round(psutil.virtual_memory().total / (1024**3), 2), "GB")

print("\nStorage Information")
disk = psutil.disk_usage("/")
print("Total storage:", round(disk.total / (1024**3), 2), "GB")
print("Free storage:", round(disk.free / (1024**3), 2), "GB")


Python version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Operating system: Linux-6.6.159+-x86_64-with-glibc2.39

CPU Information
CPU cores: 2
RAM: 12.67 GB

Storage Information
Total storage: 107.72 GB
Free storage: 107.62 GB


In [1]:

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())

    props = torch.cuda.get_device_properties(0)
    print("GPU memory:", round(props.total_memory / (1024**3), 2), "GB")
else:
    print("No GPU allocated to this runtime.")


PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU count: 1
GPU memory: 14.56 GB


In [2]:
! pip install -q gradio

In [3]:

import gradio as gr
import torch
import psutil
import platform

def get_system_info():
    # CPU information
    cpu_cores = psutil.cpu_count(logical=True)
    cpu_usage = psutil.cpu_percent(interval=1)

    # RAM information
    ram = psutil.virtual_memory()
    ram_total = round(ram.total / (1024**3), 2)
    ram_used = round(ram.used / (1024**3), 2)

    # GPU information
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_memory = torch.cuda.get_device_properties(0).total_memory
        gpu_memory = round(gpu_memory / (1024**3), 2)

        allocated = torch.cuda.memory_allocated(0)
        allocated = round(allocated / (1024**3), 3)

        gpu_status = "CUDA Available"
    else:
        gpu_name = "No GPU detected"
        gpu_memory = 0
        allocated = 0
        gpu_status = "CUDA Unavailable"

    return f"""
    ## Cloud Resource Report

    **Operating System:** {platform.system()}

    ### CPU
    - CPU Cores: {cpu_cores}
    - CPU Usage: {cpu_usage}%

    ### System Memory
    - Total RAM: {ram_total} GB
    - Used RAM: {ram_used} GB

    ### GPU
    - GPU: {gpu_name}
    - GPU Memory: {gpu_memory} GB
    - PyTorch Allocated Memory: {allocated} GB
    - Status: {gpu_status}
    """

print("Monitoring function created successfully!")


Monitoring function created successfully!


In [4]:

import gradio as gr

with gr.Blocks(title="GPU Cloud Resource Monitor") as app:

    gr.Markdown("""
    # GPU Cloud Resource Monitor
    ### Google Colab Cloud Infrastructure Dashboard

    Monitor the CPU, system memory and GPU resources
    allocated to the current cloud runtime.
    """)

    refresh_btn = gr.Button("Refresh Resource Information")

    output = gr.Markdown()

    refresh_btn.click(
        fn=get_system_info,
        inputs=None,
        outputs=output
    )

app.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9b74f01b3a702e0069.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [5]:

import torch
import time

def run_gpu_benchmark():
    if not torch.cuda.is_available():
        return "CUDA GPU is not available."

    device = torch.device("cuda")

    # Create two matrices on the GPU
    A = torch.randn(2000, 2000, device=device)
    B = torch.randn(2000, 2000, device=device)

    # Warm-up
    C = torch.matmul(A, B)
    torch.cuda.synchronize()

    # Measure execution time
    start = time.perf_counter()

    C = torch.matmul(A, B)

    torch.cuda.synchronize()
    end = time.perf_counter()

    elapsed = (end - start) * 1000

    return f"""
    ## GPU Benchmark Results

    - GPU: {torch.cuda.get_device_name(0)}
    - Operation: Matrix Multiplication
    - Matrix dimensions: 2000 × 2000
    - Execution time: {elapsed:.3f} ms
    - Result shape: {tuple(C.shape)}
    - CUDA status: Successful
    """

print("GPU benchmark function created!")


GPU benchmark function created!


In [6]:
with gr.Blocks(title="GPU Cloud Resource Monitor") as app:

    gr.Markdown("""
    # GPU Cloud Resource Monitor

    ### Google Colab Cloud Infrastructure Dashboard

    Monitor cloud resources and execute GPU computations
    using the NVIDIA GPU allocated to this runtime.
    """)

    with gr.Tab("Resource Monitoring"):

        gr.Markdown("### CPU, RAM, Storage and GPU Information")

        monitor_button = gr.Button(
            "Refresh Resource Information"
        )

        resource_output = gr.Markdown()

        monitor_button.click(
            fn=get_system_info,
            inputs=None,
            outputs=resource_output
        )

    with gr.Tab("GPU Benchmark"):

        gr.Markdown("""
        ### GPU Matrix Multiplication Benchmark

        Execute a matrix multiplication workload on the cloud GPU
        and measure its execution time.
        """)

        benchmark_button = gr.Button(
            "Run GPU Benchmark",
            variant="primary"
        )

        benchmark_output = gr.Markdown()

        benchmark_button.click(
            fn=run_gpu_benchmark,
            inputs=None,
            outputs=benchmark_output
        )

    gr.Markdown("""
    ---
    **Project:** Conjugate Gradient Performance and Scalability Analysis

    **Platform:** Google Colab Free
    """)

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://08f662073a3af76060.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
